# VAEモデルの学習率スケジューラ

このノートブックでは、Variational Autoencoder (VAE) の学習に学習率スケジューラを導入して、トレーニング中に学習率を動的に調整する方法を示します。

In [ ]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np

## 利用可能な学習率スケジューラ

PyTorchには様々な学習率スケジューラが用意されています。VAEモデルに適用できる主なスケジューラは以下の通りです：

1. **StepLR**: 一定のエポック数ごとに学習率を一定の割合で減少させる
2. **MultiStepLR**: 指定した複数のエポックで学習率を減少させる
3. **ExponentialLR**: 各エポックごとに学習率を指数関数的に減少させる
4. **CosineAnnealingLR**: 余弦関数に従って学習率を周期的に変動させる
5. **ReduceLROnPlateau**: 検証損失が改善しない場合に学習率を減少させる

ここでは、VAEモデルに最も適していると考えられるいくつかの学習率スケジューラの実装方法を示します。

## 1. StepLRスケジューラ

一定のエポック数ごとに学習率をガンマ倍に減少させます。例えば、`step_size=30`、`gamma=0.1`の場合、30エポックごとに学習率が1/10になります。

In [ ]:
# オプティマイザの初期化
initial_lr = 1e-3
optimizer = optim.Adam(model.parameters(), lr=initial_lr, weight_decay=1e-5)

# StepLRスケジューラの設定
step_size = 30  # 30エポックごとに学習率を減少
gamma = 0.1     # 学習率を1/10に減少
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

# トレーニングループ内で使用する例
for epoch in range(num_epochs):
    # トレーニングコード...
    
    # エポック終了後にスケジューラを更新
    scheduler.step()
    
    # 現在の学習率をログに記録
    current_lr = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch}, Current LR: {current_lr:.6f}')

## 2. ReduceLROnPlateauスケジューラ

検証損失が一定のエポック数にわたって改善しない場合に学習率を減少させます。早期停止（Early Stopping）と組み合わせて使用すると効果的です。

In [ ]:
# オプティマイザの初期化
initial_lr = 1e-3
optimizer = optim.Adam(model.parameters(), lr=initial_lr, weight_decay=1e-5)

# ReduceLROnPlateauスケジューラの設定
patience = 5      # 5エポック改善がなければ学習率を減少
factor = 0.2      # 学習率を1/5に減少
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',          # 損失を最小化するモード
    factor=factor,       # 減少させる割合
    patience=patience,   # 何エポック待つか
    verbose=True,        # 学習率の変更時にメッセージを表示
    min_lr=1e-6          # 最小学習率
)

# トレーニングループ内で使用する例
for epoch in range(num_epochs):
    # トレーニングコード...
    train_loss = train_epoch(model, train_loader, optimizer, device, beta)
    val_loss = validate_epoch(model, val_loader, device, beta)
    
    # 検証損失に基づいてスケジューラを更新
    scheduler.step(val_loss)
    
    # 現在の学習率をログに記録
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch}, Current LR: {current_lr:.6f}')

## 3. CosineAnnealingLRスケジューラ

余弦関数に従って学習率を周期的に変動させます。局所的な最適解から抜け出す能力を持ち、より良い解を見つける可能性があります。

In [ ]:
# オプティマイザの初期化
initial_lr = 1e-3
optimizer = optim.Adam(model.parameters(), lr=initial_lr, weight_decay=1e-5)

# CosineAnnealingLRスケジューラの設定
T_max = 10  # 余弦周期の長さ（エポック数）
eta_min = 1e-6  # 最小学習率
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=T_max,
    eta_min=eta_min
)

# トレーニングループ内で使用する例
for epoch in range(num_epochs):
    # トレーニングコード...
    
    # エポック終了後にスケジューラを更新
    scheduler.step()
    
    # 現在の学習率をログに記録
    current_lr = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch}, Current LR: {current_lr:.6f}')

## 4. 学習率スケジューラの可視化

異なる学習率スケジューラがどのように学習率を変化させるかを可視化してみましょう。

In [ ]:
def plot_lr_scheduler(scheduler_name, scheduler_fn, optimizer, num_epochs=100):
    """学習率スケジューラの挙動を可視化する関数"""
    
    # 各エポックでの学習率を記録
    lrs = []
    
    # スケジューラの初期化
    scheduler = scheduler_fn(optimizer)
    
    # 各エポックでの学習率をシミュレート
    for epoch in range(num_epochs):
        lrs.append(optimizer.param_groups[0]['lr'])
        
        if scheduler_name == "ReduceLROnPlateau":
            # ReduceLROnPlateauの場合は、検証損失が改善しないと仮定
            # 最初の20エポックは改善、その後は改善なしと仮定
            if epoch < 20:
                val_loss = 1.0 - epoch * 0.02  # 損失が減少
            elif epoch % 20 < 10:
                val_loss = 0.6  # 損失が停滞
            else:
                val_loss = 0.5 - (epoch % 10) * 0.01  # 損失が再び減少
                
            scheduler.step(val_loss)
        else:
            scheduler.step()
    
    return lrs

# 比較する学習率スケジューラ
schedulers = {
    "StepLR": lambda opt: optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.1),
    "MultiStepLR": lambda opt: optim.lr_scheduler.MultiStepLR(opt, milestones=[30, 60, 80], gamma=0.1),
    "ExponentialLR": lambda opt: optim.lr_scheduler.ExponentialLR(opt, gamma=0.97),
    "CosineAnnealingLR": lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20, eta_min=1e-6),
    "ReduceLROnPlateau": lambda opt: optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.1, patience=5, verbose=False, min_lr=1e-6
    )
}

# 可視化
plt.figure(figsize=(12, 8))

for name, scheduler_fn in schedulers.items():
    # 各スケジューラごとに新しいオプティマイザを作成
    dummy_model = torch.nn.Linear(10, 1)  # ダミーモデル
    optimizer = optim.Adam(dummy_model.parameters(), lr=1e-3)
    
    # 学習率の変化をシミュレート
    lrs = plot_lr_scheduler(name, scheduler_fn, optimizer)
    
    # プロット
    plt.plot(lrs, label=name)

plt.xlabel('エポック')
plt.ylabel('学習率')
plt.title('各学習率スケジューラの挙動比較')
plt.legend()
plt.grid(True)
plt.yscale('log')  # 対数スケールで表示
plt.show()

## 5. VAEモデルに学習率スケジューラを適用する方法

既存のVAEモデルのトレーニングコードに学習率スケジューラを追加するには、以下のようにコードを変更します。ここでは、検証損失に基づいて学習率を調整する`ReduceLROnPlateau`スケジューラを追加する例を示します。

In [ ]:
# モデルの初期化部分
input_dim = X_scaled.shape[1]
hidden_dim = 1024
latent_dim = 64
batch_size = 32
learning_rate = 1e-3
beta = 0.5

# モデルとオプティマイザの初期化
model = WaveletVAE(input_dim, hidden_dim, latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)

# 学習率スケジューラの追加
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',          # 損失を最小化するモード
    factor=0.2,          # 学習率を0.2倍（1/5）に減少
    patience=3,          # 3エポック改善がなければ減少
    verbose=True,        # 学習率の変更時にメッセージを表示
    min_lr=1e-6          # 最小学習率
)

# トレーニングループ
num_epochs = 100
train_losses = []
val_losses = []
train_recon_losses = []
val_recon_losses = []
train_kld_losses = []
val_kld_losses = []
learning_rates = []  # 学習率の履歴を記録

best_val_loss = float('inf')
patience = 10
patience_counter = 0

print("トレーニング開始...")
for epoch in range(num_epochs):
    train_loss, train_recon, train_kld = train_epoch(model, train_loader, optimizer, device, beta)
    val_loss, val_recon, val_kld = validate_epoch(model, val_loader, device, beta)
    
    # 現在の学習率を記録
    current_lr = optimizer.param_groups[0]['lr']
    learning_rates.append(current_lr)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_recon_losses.append(train_recon)
    val_recon_losses.append(val_recon)
    train_kld_losses.append(train_kld)
    val_kld_losses.append(val_kld)
    
    # 学習率スケジューラを更新
    scheduler.step(val_loss)
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'Train Recon: {train_recon:.4f}, Train KLD: {train_kld:.4f}, LR: {current_lr:.6f}')
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # ベストモデルを保存
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),  # スケジューラの状態も保存
            'loss': val_loss,
        }, 'best_vae_model.pth')
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch}')
        break

print("トレーニング完了!")

# 学習率の変化をプロット
plt.figure(figsize=(10, 5))
plt.plot(learning_rates)
plt.xlabel('エポック')
plt.ylabel('学習率')
plt.title('トレーニング中の学習率の変化')
plt.grid(True)
plt.yscale('log')  # 対数スケールで表示
plt.show()

## 6. 効果的な学習率スケジューリングのコツ

VAEモデルの学習において効果的な学習率スケジューリングのコツをいくつか紹介します：

1. **初期学習率の選択**: 初期学習率は小さすぎると学習が遅く、大きすぎると収束しない可能性があります。1e-3から1e-4の範囲が一般的に良い出発点となります。

2. **温かい開始（Warm Start）**: 最初の数エポックは低い学習率から始めて徐々に上げる「温かい開始」を試すこともできます。これにより初期の不安定さを抑えることができます。

3. **周期的な学習率**: `CosineAnnealingLR`のような周期的な学習率スケジューラを使用すると、局所的な最小値から抜け出しやすくなります。

4. **プラトー検出**: `ReduceLROnPlateau`は特にVAEのような複雑なモデルで有効です。損失が改善しなくなったときに自動的に学習率を下げることで、より細かい調整が可能になります。

5. **最小学習率の設定**: 学習率があまりに小さくなると効果的な学習ができなくなるため、最小学習率（例えば1e-6）を設定することが重要です。

## 7. まとめ

VAEモデルのトレーニングにおいて学習率は非常に重要なハイパーパラメータです。適切な学習率スケジューラを選択することで、以下のような利点があります：

1. **より早い収束**: 適切な学習率調整により、少ないエポック数で良い結果を得ることができます。

2. **より良い性能**: 局所的な最小値を回避し、よりグローバルな最小値に到達できる可能性が高まります。

3. **安定したトレーニング**: 急激な勾配の変化による不安定さを抑えることができます。

VAEモデルに最も適した学習率スケジューラは、通常は`ReduceLROnPlateau`や`CosineAnnealingLR`です。これらのスケジューラは、モデルの学習状況に応じて学習率を動的に調整し、効率的なトレーニングを可能にします。